# OpenAI + QDrant

In [1]:
import os
import re
from tqdm import tqdm
from utils import *
import logging
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# set up a logger with file of current time
logging.basicConfig(filename=f'../data/logs/{datetime.now().strftime("%Y-%m-%d_%H-%M-%S")}.log', level=logging.DEBUG)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Felix\\code\\trustbit\\enterprise_rag_challenge_fk\\data\\logs\\2025-03-04_20-37-57.log'

In [5]:
# company = "reit"
company = "TSX_Y_2022"
# company = "NASDAQ_CLXT_2022"

test_pdf_path = f"../data/dev/pdfs/{company}.pdf"

In [2]:
# Load questions
with open(f"../data/round2/questions.json", "r") as file:
    questions = json.load(file)

# open companies list
# companies_dict = get_companies_dict(r"C:\Users\felix.krause\code\trustbit\enterprise-rag-challenge\dataset_v2.json")
companies_dict = get_companies_dict(r"../data/round2/subset.json")

In [3]:
# Load prompt
with open("prompts/prompt_openAI-qdrant.md", "r") as file:
    system_prompt = file.read()

print(system_prompt[:100])

## SYSTEM PROMPT

You are a chatbot designed to answer questions about company annual reports. The i


In [7]:
# only keep subset for dev
companies = ["Ziff Davis, Inc.", ""]
companies_dict = {company: data for company, data in companies_dict.items() if company in companies}

In [18]:
companies_dict

{'ACRES Commercial Realty Corp.': {'name': 'ACRES Commercial Realty Corp.',
  'sha1': '0279901b645e568591ad95dac2c2bf939ef0c00d',
  'id': None},
 'Aptevo Therapeutics Inc.': {'name': 'Aptevo Therapeutics Inc.',
  'sha1': '0981826b4b43a88920f3e01c71ae73539bab84cc',
  'id': None},
 'Downer EDI Limited': {'name': 'Downer EDI Limited',
  'sha1': '0a61a353b1ea9fd9b8f63b60239634ca3007d58f',
  'id': None},
 'Odyssey Gold Limited': {'name': 'Odyssey Gold Limited',
  'sha1': '0c0faea14d108e1617f2d6d2a7c1aae04eb88fe0',
  'id': None},
 'NextNav Inc.': {'name': 'NextNav Inc.',
  'sha1': '0f111d244aee3d976684995a222fa177a64571c4',
  'id': None},
 'Peako Limited': {'name': 'Peako Limited',
  'sha1': '105688726e097505beef4934896193ac51295037',
  'id': None},
 'Mosaic Brands Limited': {'name': 'Mosaic Brands Limited',
  'sha1': '12bff07b957b1c8f8cad9d917ca18005720cce9b',
  'id': None},
 'Aurora Innovation, Inc.': {'name': 'Aurora Innovation, Inc.',
  'sha1': '13999998018cc53440310d94a26d1e8957e2277f',

## Docling - Processing PDFs

In [ ]:
# Potential for parallelization for files?
import os
from docling.document_converter import DocumentConverter
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm


converter = DocumentConverter()  # 20 sec for setting up

preproc_logs = {"failed": [], "skipped": []}


def process_company(company, data):
    pdf_path = f"../data/round2/pdfs/{data['sha1']}.pdf"
    docling_md_path = f"../data/docling_md/round2/docling_{data['sha1']}.md"

    if not os.path.exists(docling_md_path):
        print(f"Processing {company}")
        try:
            result = converter.convert(pdf_path)
            with open(docling_md_path, "w", encoding="utf-8") as file:
                file.write(result.document.export_to_markdown())
            # Return None error to indicate success.
            return (company, None)
        except Exception as e:
            print(f"Error processing {company}: {e}")
            return (company, e)
    else:
        print(f"Skipping {company}")
        return (company, "skipped")

# Use ThreadPoolExecutor to process companies concurrently.
with ThreadPoolExecutor(max_workers=1) as executor:
    # Submit all tasks
    futures = {executor.submit(process_company, company, data): company
               for company, data in companies_dict.items()}

    # Process tasks as they complete.
    for future in tqdm(as_completed(futures), total=len(futures)):
        company, result = future.result()
        if result is None:
            continue  # Successful processing.
        elif result == "skipped":
            preproc_logs["skipped"].append(company)
        else:
            preproc_logs["failed"].append({"company_id": company, "error": result})

if preproc_logs["failed"]:
    print(f"Nr of failed companies: {len(preproc_logs['failed'])}")


# RUNTIMES:
# REIT: execution for 7.4MB with 97 pages -> 5 min
# Yellow Pages: execution for 1.4MB with 77 pages -> 4 min
# Calyxt: execution for 0.7MB with 88 pages -> 3min
# TODO check on google colab

# NOTES:
# brackets in tables indicate negative values
# parsing issues (see current system prompt)

Skipping ACRES Commercial Realty Corp.Skipping Aptevo Therapeutics Inc.

Skipping Downer EDI Limited
Skipping Odyssey Gold Limited
Skipping Peako Limited
Skipping NextNav Inc.
Skipping Mosaic Brands Limited
Skipping Aurora Innovation, Inc.
Skipping Crombie REIT
Skipping Medallion Financial Corp.
Skipping OFX Group Limited
Skipping Enact Holdings, Inc.
Skipping FNCB Bancorp, Inc.
Skipping BetMakers Technology Group Ltd
Skipping Celldex Therapeutics, Inc.
Skipping SIG plc
Skipping Motability Operations Group plc
Skipping BCB Bancorp, Inc.
Skipping 1-800-FLOWERS.COM, INC.
Skipping Weis Markets, Inc.
Skipping Odyssey Group Holdings, Inc.
Skipping Rectifier Technologies Ltd
Skipping Origin Bancorp, Inc.
Skipping Blue Apron Holdings, Inc.
Skipping Structural Monitoring Systems Plc
Skipping Ocugen, Inc.
Skipping Guaranty Bancshares, Inc.
Skipping Infinera Corporation
Skipping Liberty Broadband Corporation
Skipping Incyte Corporation
Processing MainStreet Bancshares, Inc.
Skipping Starvest plc

  0%|          | 0/100 [00:00<?, ?it/s]

Skipping Galileo Resources Plc
Processing SThree plc
Skipping Incitec Pivot Limited
Processing EPI (HOLDINGS) LIMITED
Processing Sonic Automotive, Inc.


 39%|███▉      | 39/100 [14:38<22:53, 22.52s/it]

Processing Commerzbank


...

Processing Seiko Epson Corporation


 41%|████      | 41/100 [42:36<1:33:29, 95.08s/it]

Processing DiaMedica Therapeutics Inc.


 42%|████▏     | 42/100 [56:34<2:18:00, 142.77s/it]

Processing James Halstead plc


In [22]:
# STORE as md with correct utf-8 encoding
# with open(f"../data/docling_md/dev/docling_{company}.md", "w", encoding="utf-8") as file:
#     file.write(result.document.export_to_markdown())

## Custom Advanced Chunking

In [6]:
def is_table_paragraph(paragraph):
    """
    Returns True if the paragraph looks like a markdown table block.
    We assume that a table block consists of lines that begin with '|'
    and that at least one line is a separator (i.e. contains a series of dashes).
    """
    lines = paragraph.strip().splitlines()
    if not lines:
        return False
    if not all(line.lstrip().startswith('|') for line in lines):
        return False
    for line in lines:
        if re.search(r'\|\s*-{2,}', line):
            return True
    return False

def get_overlap_text(text, desired_overlap, max_overlap=None):
    """
    Extracts an overlap string from the end of `text` made up of whole words.

    Args:
        text (str): The text from which to extract the overlap.
        desired_overlap (int): Desired minimum number of characters to include in the overlap.
        max_overlap (int or None): Maximum allowed overlap (in characters). If None, no hard cap is applied.

    Returns:
        str: A string composed of whole words from the end of `text` whose total length
             is at least `desired_overlap` (if possible) but not exceeding `max_overlap` (if provided).
    """
    if max_overlap is None:
        max_overlap = desired_overlap * 2

    words = text.split()
    if not words:
        return ""

    overlap_words = []
    total_length = 0
    # Iterate over the words in reverse order.
    for word in reversed(words):
        # Add a space before each word except the first one we add.
        addition_length = len(word) if not overlap_words else len(word) + 1
        # If a maximum overlap is set and adding this word would exceed it,
        # then stop (unless no word has been added yet).
        if max_overlap is not None and total_length + addition_length > max_overlap:
            if overlap_words:
                break
            # Else, if even a single word is longer than max_overlap, add its characters until max is reached.
            if len(word) > max_overlap:
                overlap_words.append(word[:max_overlap - total_length])
                break
        overlap_words.append(word)
        total_length += addition_length
        # Stop if we've reached at least the desired overlap.
        if total_length >= desired_overlap:
            break

    # Reassemble the words in the correct order.
    return " ".join(reversed(overlap_words))

def split_paragraph(paragraph, max_chars, overlap_chars=0, max_overlap_chars=None):
    """
    Splits a non-table paragraph (by words) so that each part is below max_chars.
    If the paragraph is split into multiple parts, each subsequent part will begin with an
    overlap taken from the tail of the previous part. The overlap is composed of whole words,
    trying to reach at least `overlap_chars` characters but never exceeding `max_overlap_chars` (if set).

    Args:
        paragraph (str): The paragraph text to split.
        max_chars (int): Maximum allowed character length per part.
        overlap_chars (int): Desired number of characters to overlap between parts.
        max_overlap_chars (int or None): Absolute maximum allowed overlap (in characters).

    Returns:
        list of str: A list of paragraph parts, each no longer than max_chars.
    """
    words = paragraph.split()
    parts = []
    current = ""

    for word in words:
        candidate = (current + " " + word).strip() if current else word
        if len(candidate) > max_chars:
            # If we have accumulated some text, finalize it.
            if current:
                parts.append(current)
                # Compute an overlap from the end of current, as whole words.
                overlap_text = get_overlap_text(current, overlap_chars, max_overlap_chars) if overlap_chars > 0 else ""
                # Try to start the next segment with the overlap plus the current word.
                new_candidate = (overlap_text + " " + word).strip() if overlap_text else word
                if overlap_text and len(new_candidate) <= max_chars:
                    current = new_candidate
                else:
                    current = word
            else:
                # In the unlikely case a single word exceeds max_chars, yield it as its own part.
                parts.append(word)
                current = ""
        else:
            current = candidate
    if current:
        parts.append(current)
    return parts

def split_content(text, max_chars, overlap_chars=0, max_overlap_chars=None):
    """
    Split a block of text into parts no longer than max_chars.
    The text is first split into paragraphs (using double newlines) so that
    tables and other contiguous blocks remain intact. If an individual paragraph
    is too long and is not a table, it is further split with an overlap between parts.
    Also, for each final part a flag is computed (True/False) indicating whether the part is a table.

    Returns:
        tuple:
            - parts (list of str): The split parts.
            - is_table_ls (list of bool): For each part, True if it is a table block, False otherwise.
    """
    paragraphs = text.split("\n\n")
    parts = []
    is_table_ls = []  # This will store a True/False flag per part.

    current_part = ""
    current_part_is_table = None  # Will be set to True for table parts, False for non-table.

    for para in paragraphs:
        para = para.strip()
        if not para:
            continue

        para_is_table = is_table_paragraph(para)
        # If this paragraph is a table, do not merge with other content.
        if para_is_table:
            # TODO also include overlap to table!
            # TODO if table is larger then embedding model allows, split into smaller tables - how often does it happen?
            if current_part:
                parts.append(current_part)
                is_table_ls.append(current_part_is_table)
                current_part = ""
                current_part_is_table = None
            parts.append(para)
            is_table_ls.append(True)
            continue

        # For non-table paragraphs, if too long, split further.
        if len(para) > max_chars:
            sub_parts = split_paragraph(para, max_chars, overlap_chars, max_overlap_chars)
        else:
            sub_parts = [para]

        for sub in sub_parts:
            # If current_part is non-empty but from a different type (table vs. non-table), flush it.
            if current_part and current_part_is_table is not False:
                parts.append(current_part)
                is_table_ls.append(current_part_is_table)
                current_part = ""
                current_part_is_table = False

            candidate = (current_part + "\n\n" + sub).strip() if current_part else sub
            if len(candidate) <= max_chars:
                current_part = candidate
                current_part_is_table = False  # Non-table content.
            else:
                if current_part:
                    parts.append(current_part)
                    is_table_ls.append(current_part_is_table)
                current_part = sub
                current_part_is_table = False
        prev_para = para

    if current_part:
        parts.append(current_part)
        is_table_ls.append(current_part_is_table)

    return parts, is_table_ls

def chunk_markdown(markdown_text, document_name, context_window=2, max_chunk_size=2000, overlap_chars=500, max_overlap_chars=None):
    """
    Splits a markdown document into chunks based on headings and attaches metadata,
    including previous context_window section titles. If a section's content is too large,
    it is further split into sub‐chunks with an optional overlap (in characters) between
    consecutive parts.

    Args:
        markdown_text (str): The markdown document.
        context_window (int): Number of previous headings to include as context.
        max_chunk_size (int): Maximum allowed character length per (sub-)chunk.
        overlap_chars (int): Desired number of characters to overlap when splitting paragraphs (considering words).
        max_overlap_chars (int): Maximum allowed overlap between consecutive parts.

    Returns:
        list of dict: Each dict represents a (sub-)chunk and contains:
            - 'title': Heading text.
            - 'level': Heading level (number of '#' characters).
            - 'content': Content of the (sub-)chunk.
            - 'previous_titles': List of previous section titles.
            - 'part_index': Index of this part (1-based).
            - 'total_parts': Total number of parts for this section.
    """
    heading_pattern = re.compile(r'^(#{1,6})\s+(.*)$', re.MULTILINE)
    matches = list(heading_pattern.finditer(markdown_text))

    chunks = []

    # If no headings are found, treat the whole document as one chunk.
    if not matches:
        content_parts, is_table_ls = split_content(markdown_text.strip(), max_chunk_size, overlap_chars, max_overlap_chars)
        assert len(content_parts) == len(is_table_ls)
        total_parts = len(content_parts)
        prev_part = ""
        for i, (part, is_table) in enumerate(zip(content_parts, is_table_ls), start=1):
            if is_table:
                part = prev_part + "\n\n" + part  # Add a newline after the table to separate it from the next part.
            chunks.append({
                'title': None,
                'level': None,
                'table': is_table,
                'content': part,
                'previous_titles': [],
                'part_index': i,
                'total_parts': total_parts
                # TODO also include page number of chunk here
            })
            prev_part = part
        return chunks

    # Process each section defined by a heading.
    for i, match in enumerate(matches):
        heading_marker = match.group(1)
        title = match.group(2).strip()
        level = len(heading_marker)

        # Determine content boundaries.
        content_start = match.end()
        content_end = matches[i+1].start() if i+1 < len(matches) else len(markdown_text)
        content = markdown_text[content_start:content_end].strip()

        previous_titles = [m.group(0).strip() for m in matches[max(0, i-context_window):i]]
        next_titles = [m.group(0).strip() for m in matches[i+1:i+context_window+1]]

        # Split the section's content if it's too large.
        content_parts, is_table_ls = split_content(content, max_chunk_size, overlap_chars, max_overlap_chars)
        assert len(content_parts) == len(is_table_ls), f"Lengths: {len(content_parts)} vs {len(is_table_ls)}"
        total_parts = len(content_parts)
        prev_part = ""
        for j, (part, is_table) in enumerate(zip(content_parts, is_table_ls), start=1):
            if is_table:
                try:
                    if len(chunks[-1]["content"]) < 100 and j > 1:
                        addition = chunks[-2]["content"] + chunks[-1]["content"]
                    else:
                        addition = chunks[-1]["content"]
                    part = addition + "\n\n" + part
                except:
                    pass

            chunk = {
                'title': title,
                'level': level,
                'previous_titles': previous_titles,
                'next_titles': next_titles,
                'company': document_name,
                'table': is_table,
                'part_index': j,
                'total_parts': total_parts,
                'content': part,
            }
            chunks.append(chunk)
            prev_part = part
    return chunks

def pretty_print_chunks(chunks):
    for chunk in chunks:
        print("Title:", chunk['title'])
        print("Company:", chunk['company'])
        print("Level:", chunk['level'])
        print("Is Table:", chunk['table'])
        print("Part: {}/{}".format(chunk['part_index'], chunk['total_parts']))
        print("Previous Titles:", chunk['previous_titles'])
        print("Next Titles:", chunk['next_titles'])
        print("Content:\n", chunk['content'])
        print("-" * 40)

# --- Example usage ---
sample_markdown = """
# Annual Report 2024

Introductory text for the report. This section contains some long descriptive text that might be very large and thus needs to be split into parts. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Vestibulum vel dolor eget velit efficitur aliquet. Donec ullamcorper, metus ac convallis facilisis, lorem urna commodo nunc, nec facilisis velit leo et augue. Sed sodales, purus eget vulputate mollis, enim ipsum convallis magna, eget interdum urna arcu sit amet libero.

## Financial Overview

| For the years ended December 31,                                                                  | 2022         | 2021         |
|---------------------------------------------------------------------------------------------------|--------------|--------------|
| Cash flows from operating activities                                                              |              |              |
| Change in operating assets and liabilities                                                        | $  4,728     | $  26,154    |
| Stock-based compensation cash payments                                                            | (7,155)      | (5,541)      |
| Funding of post-employment benefit plans in excess of costs                                       | (32,106)     | (7,523)      |
| Income taxes paid, net                                                                            | (7,758)      | -            |
| Cash flows from operations, excluding the above                                                   | 91,791       | 91,489       |
|                                                                                                   | $  49,500    | $  104,579   |
| Cash flows used in investing activities                                                           |              |              |
| Additions to intangible assets                                                                    | $  (4,911)   | $  (4,957)   |
| Additions to property and equipment                                                               | (93)         | (117)        |
| Payments received from net investment in subleases                                                | 1,338        | 593          |
|                                                                                                   | $  (3,666)   | $  (4,481)   |
| Cash flows used in financing activities                                                           |              |              |
| Repayment of exchangeable debentures                                                              | $  -         | $  (107,033) |
| Repurchase of common shares through NCIBs                                                         | (12,404)     | (5,334)      |
| Repurchase of common shares per plan of arrangement, net of treasury shares and transaction costs | (96,125)     | -            |
| Issuance of common shares                                                                         | 153          | 111          |
| Payment of lease obligations                                                                      | (2,947)      | (3,045)      |
| Dividends paid                                                                                    | (14,163)     | (14,730)     |
|                                                                                                   | $  (125,486) | $  (130,031) |
| NET DECREASE IN CASH                                                                              | $  (79,652)  | $  (29,933)  |
| CASH, BEGINNING OF YEAR                                                                           | 123,559      | 153,492      |
| CASH, END OF YEAR                                                                                 | $  43,907    | $  123,559   |

Additional commentary follows the table to elaborate on the financial performance. More descriptive text is added to simulate a very large chunk that should be split intelligently. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Praesent ac nibh vestibulum, imperdiet orci sit amet, dictum urna.
"""

# For demonstration, set a lower max_chunk_size (e.g. 500 characters) and an overlap of 50 characters.
chunks = chunk_markdown(sample_markdown, company, context_window=3, max_chunk_size=200, overlap_chars=50, max_overlap_chars=100)
#
pretty_print_chunks(chunks)

# TODO TODO TODO
# TODO if table, add chunk before too
# TODO merge with next section if a chunk is too small

Title: Annual Report 2024
Company: TSX_Y_2022
Level: 1
Is Table: False
Part: 1/3
Previous Titles: []
Next Titles: ['## Financial Overview']
Content:
 Introductory text for the report. This section contains some long descriptive text that might be very large and thus needs to be split into parts. Lorem ipsum dolor sit amet, consectetur adipiscing
----------------------------------------
Title: Annual Report 2024
Company: TSX_Y_2022
Level: 1
Is Table: False
Part: 2/3
Previous Titles: []
Next Titles: ['## Financial Overview']
Content:
 Lorem ipsum dolor sit amet, consectetur adipiscing elit. Vestibulum vel dolor eget velit efficitur aliquet. Donec ullamcorper, metus ac convallis facilisis, lorem urna commodo nunc, nec facilisis
----------------------------------------
Title: Annual Report 2024
Company: TSX_Y_2022
Level: 1
Is Table: False
Part: 3/3
Previous Titles: []
Next Titles: ['## Financial Overview']
Content:
 convallis facilisis, lorem urna commodo nunc, nec facilisis velit leo et a

In [14]:
# Chunking example
company_test = "NASDAQ_CLXT_2022"

with open(f"../data/docling_md/dev/docling_{company_test}.md", "r", encoding="utf-8") as file:
    text = file.read()

sample = text[:10000]

chunks = chunk_markdown(sample, company, context_window=3, max_chunk_size=2000, overlap_chars=500)

pretty_print_chunks(chunks[:10])

Title: UNITED STATES SECURITIES AND EXCHANGE COMMISSION
Company: TSX_Y_2022
Level: 2
Is Table: False
Part: 1/1
Previous Titles: []
Next Titles: ['## FORM 10-K', '## Calyxt, Inc.', '## DOCUMENTS INCORPORATED BY REFERENCE']
Content:
 Washington, D.C. 20549
----------------------------------------
Title: FORM 10-K
Company: TSX_Y_2022
Level: 2
Is Table: False
Part: 1/1
Previous Titles: ['## UNITED STATES SECURITIES AND EXCHANGE COMMISSION']
Next Titles: ['## Calyxt, Inc.', '## DOCUMENTS INCORPORATED BY REFERENCE', '## Table of Contents']
Content:
 <!-- image -->

(Mark One)

☒

ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF

1934

For the fiscal year ended December 31, 2022;

or

☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934

For the transition period from                       to

Commission file number 001-38161
----------------------------------------
Title: Calyxt, Inc.
Company: TSX_Y_2022
Level: 2
Is Table: 

In [9]:
len(chunks)

389

## Create embeddings

In [7]:
# OPENAI EMBEDDINGS
from openai import OpenAI

embedding_client = OpenAI()

# Max input: 8191 tokens

# embedding_model = "text-embedding-3-large"
# embedding_size = 3072

embedding_model = "text-embedding-3-small"
embedding_size = 1536

In [ ]:
# Create embeddings for single sample
contents = [f"{chunk['title']}: {chunk['content']}" for chunk in chunks]
contents_cleaned = [txt.replace("\n", " ") for txt in contents]
embeddings = embedding_client.embeddings.create(input=contents_cleaned, model=embedding_model).data

embeddings = [emb.embedding for emb in embeddings]

# attach to chunks
db_data = []
for idx, chunk in enumerate(chunks):
    db_data.append({
        "embedding": embeddings[idx],
        "metadata": chunk,
    })

chunk.keys()

In [18]:
# hotfix
def clean_chunks(chunks, max_chars=10000):
    new_chunks = []
    for chunk in chunks:
        if chunk["content"].count("TimesNewRoman") > 4:  # to clean weird 'MongoDB, Inc.' chunks
            continue
        if "<!-- image -->".strip() in chunk["content"].strip():  # 'Poste Italiane' edge case
            continue
        if chunk["content"].strip():
            if len(chunk["content"]) > max_chars:
                chunk["content"] = chunk["content"][:max_chars]
            new_chunks.append(chunk)
    return new_chunks

In [19]:
company = "Poste Italiane"
company_id = companies_dict[company]["sha1"]

with open(f"../data/docling_md/round2/docling_{company_id}.md", "r", encoding="utf-8") as file:
                text = file.read()

chunks = chunk_markdown(text, company, context_window=4, max_chunk_size=3000, overlap_chars=500)

In [39]:
# EMBED IN BULK
from openai import OpenAI
embedding_client = OpenAI()

# db_data = []
failed_companies = []

for company, data in tqdm(companies_dict.items()):
    if company in ["Poste Italiane"]:
        try:
            company_id = data["sha1"]

            with open(f"../data/docling_md/round2/docling_{company_id}.md", "r", encoding="utf-8") as file:
                text = file.read()

            chunks = chunk_markdown(text, company, context_window=4, max_chunk_size=3000, overlap_chars=500)
            chunks = clean_chunks(chunks, max_chars=15000)
            contents = [f"{chunk['title']}: {chunk['content']}" for chunk in chunks]
            contents_cleaned = [txt.replace("\n", " ") for txt in contents]

            # FOR DEBUGGING
            # embeddings_collected = []
            # for chunk in tqdm(reversed(contents_cleaned)):
            #     try:
            #         embeddings = embedding_client.embeddings.create(input=chunk, model=embedding_model).data
            #     except:
            #         print("####", chunk)
            #     embeddings_collected.append(embeddings[0].embedding)

            embeddings = embedding_client.embeddings.create(input=contents_cleaned[1000:], model=embedding_model).data
            embeddings_collected = [emb.embedding for emb in embeddings]

            for idx, chunk in enumerate(chunks[1000:]):
                db_data.append({
                    "embedding": embeddings_collected[idx],
                    "metadata": chunk,
                })
        except Exception as e:
            print(f"Error processing {company}: {e}")
            failed_companies.append(company)

# if failed_companies:
#     failed_companies_old = failed_companies.copy()
#     print(f"WARNING: Nr of failed companies: {len(failed_companies)}")

100%|██████████| 100/100 [00:11<00:00,  8.49it/s]


In [41]:
len(db_data)

43883

In [42]:
failed_companies

[]

In [155]:
# # LOCAL APPROACH
# from sentence_transformers import SentenceTransformer
#
# # Initialize the pre-trained model
# embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
# embedding_size = 384
#
# # TODO make this more sophisticated, to include metadata in embedding
# contents = [f"{chunk['title']} - {chunk['content']}" for chunk in chunks]
# contents_cleaned = [txt.replace("\n", " ") for txt in contents]
#
# # Generate the embedding
# embeddings = embedding_model.encode(contents_cleaned)
# embeddings = [emb.tolist() for emb in embeddings]

In [43]:
db_data[1]["metadata"]

{'title': 'Portfolio Stability',
 'level': 2,
 'previous_titles': ['## Selective Growth &amp; Attractive Financing Vehicles'],
 'next_titles': ['## Capital Markets Flexibility',
  '## Corporate Responsibility',
  '## Looking Ahead',
  '## UNITED STATES SECURITIES AND EXCHANGE COMMISSION'],
 'company': 'ACRES Commercial Realty Corp.',
 'table': False,
 'part_index': 1,
 'total_parts': 1,
 'content': "The ACRES asset management team prides itself on its ability to tailor solutions that will keep borrowers on track despite challenges that may arise. Current volatility and uncertainty in the market have proved to be such challenges, but ones we've been able to manage as a result of our proactive asset management approach.\n\nThanks to our asset management team, we've reduced our watch list to 5% of the loan portfolio at the end of 2022, down from 8% at the end of 2021. In January 2023, one watch list loan, a hotel portfolio in the southwest region with a par value of $56.5 million was paid

## Creating vector database

In [50]:
# Specify collection name
# collection_name = "rag_test_db"
collection_name = "erc_2025_2"

In [51]:
# https://qdrant.tech/documentation/beginner-tutorials/search-beginners/
# Local quickstart: https://qdrant.tech/documentation/quickstart/
from qdrant_client import models, QdrantClient

# Connect to Qdrant
# db_client = QdrantClient(":memory:")

# first start qdrant server
# set up: docker pull qdrant/qdrant
# docker run -p 6333:6333 -p 6334:6334 -v "$(pwd)/data/qdrant_storage:/qdrant/storage:z" qdrant/qdrant
db_client = QdrantClient(url="http://localhost:6333")

# Define collection parameters
# embedding_size = 384

if not db_client.collection_exists(collection_name):
    db_client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
                size=embedding_size,
                distance=models.Distance.COSINE  # or Distance.EUCLID, etc.
            )
    )

db_client.upload_points(
    collection_name=collection_name,
    points=[
        models.PointStruct(id=idx, vector=data["embedding"], payload=data["metadata"])
        for idx, data in enumerate(db_data)
    ],
)

## Retrieving from vector DB

In [52]:
# https://qdrant.tech/documentation/beginner-tutorials/search-beginners/
def retriever(query, company_name, collection_name, embedding_client, db_client, k=5):
    # query_embedding = embedding_model.encode(query).tolist()
    query_embedding = embedding_client.embeddings.create(input=query, model=embedding_model).data[0].embedding
    hits = db_client.query_points(
        collection_name=collection_name,
        query=query_embedding,
        # https://qdrant.tech/documentation/concepts/filtering/
        query_filter=models.Filter(must=[models.FieldCondition(key="company", match=models.MatchValue(value=company_name))]),
        limit=k  # number of top results
    ).points

    return hits

In [57]:
# demo_query = "For which year is the report?"
query_demo = "What is chapter 3 about?"

hits = retriever(query_demo, "ACRES Commercial Realty Corp.", collection_name, embedding_client, db_client, k=3)

for hit in hits:
    print(hit.payload, "score:", hit.score)

{'title': 'ITEM 3. LEGAL PROCEEDINGS', 'level': 2, 'previous_titles': ['## General Risk Factors', '## We cannot predict the effects on us of actions taken by the U.S. government and governmental agencies in response to economic conditions in the U.S.', '## ITEM 1B. UNRESOLVED STAFF COMMENTS', '## ITEM 2. PROPERTIES'], 'next_titles': ['## ITEM 4. MINE SAFETY DISCLOSURES', '## PART II', "## ITEM 5. MARKET FOR REGISTRANT'S COMMON EQUITY, RELATED STOCKHOLDER MATTERS AND ISSUER PURCHASES OF EQUITY SECURITIES", '## Market Information'], 'company': 'ACRES Commercial Realty Corp.', 'table': False, 'part_index': 1, 'total_parts': 1, 'content': "Refer to 'Part II - Item 8. Financial Statements and Supplementary Data - Note 24 - Commitments and Contingencies' for the applicable disclosures."} score: 0.25742877
{'title': 'INDEX TO ANNUAL REPORT ON FORM 10-K', 'level': 2, 'previous_titles': ['## Trading Symbol(s)', '## Name of each exchange on which registered', '## DOCUMENTS INCORPORATED BY REFERE

In [52]:
completion = get_company_name("Did Calyxt, Inc. mention any mergers or acquisitions in the annual report", companies_dict)
print("OUTPUT:", completion)

Found company name with re: Calyxt, Inc.
{'id': 'NASDAQ_CLXT_2022.pdf', 'sha1': '40b5cfe0d7bbf59e186492bfbe1b5002d44af332'}
OUTPUT: ('Calyxt, Inc.', 'NASDAQ_CLXT_2022.pdf', '40b5cfe0d7bbf59e186492bfbe1b5002d44af332')
